# 08 - Heterogeneity Analysis

Do effects of cannabis legalization on traffic fatalities differ by state characteristics? Urban vs rural, neighboring states spillovers, alcohol culture proxies.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

DATA_DIR = Path("../data/processed")
OUT_DIR  = Path("../outputs")
OUT_DIR.mkdir(exist_ok=True)

FARS_FILE = "fars_state_year.parquet"
CDC_FILE  = "cdc_state_year.parquet"

# Load FARS panel
if not (DATA_DIR / FARS_FILE).exists():
    raise FileNotFoundError(
        f"{FARS_FILE} not found. Run:\n"
        "  python scripts/download_fars.py\n"
        "  python src/build_fars_panel.py"
    )
fars = pd.read_parquet(DATA_DIR / FARS_FILE)
leg  = pd.read_csv("../data/codebooks/state_legalization_dates.csv")
print(f"FARS panel: {fars.shape}  |  States: {fars['state'].nunique()}  |  Years: {sorted(fars['year'].dropna().unique()[:3])}...{sorted(fars['year'].dropna().unique()[-3:])}")

In [ ]:
from linearmodels.panel import PanelOLS

In [ ]:
primary = "total_fatalities_per_100k" if "total_fatalities_per_100k" in fars.columns else "total_fatalities"

# Add state characteristics for heterogeneity
# Urban share (Census urbanization 2010) — add manually or from census file
# For now flag early vs late adopters
fars_reg = fars.merge(leg[['state','retail_sales_year']], on='state', how='left')
never_treated = leg[leg['retail_sales_year'].isna()]['state'].tolist()
in_window = leg[leg['retail_sales_year'].between(2010,2022)]['state'].tolist()

fars_reg['early_adopter'] = fars_reg['retail_sales_year'].isin([2014,2015,2016,2017]).fillna(False)
fars_reg['post'] = (
    fars_reg['retail_sales_year'].notna() &
    (fars_reg['year'] >= fars_reg['retail_sales_year'])
).astype(float)
fars_reg['post_early'] = (fars_reg['post'] * fars_reg['early_adopter'].astype(float))
fars_reg['post_late']  = (fars_reg['post'] * (~fars_reg['early_adopter']).astype(float) * (fars_reg['retail_sales_year'].notna()).astype(float))

sub = fars_reg[fars_reg['state'].isin(in_window + never_treated)].copy()
idx = sub.set_index(['state','year'])

fe = PanelOLS(
    idx[primary],
    idx[['post_early','post_late']],
    entity_effects=True, time_effects=True,
).fit(cov_type='clustered', cluster_entity=True)

print("Heterogeneity by adoption timing:")
print(f"  Early adopters (2014-2017): ATT = {fe.params['post_early']:.4f}")
print(f"  Late adopters  (2018-2022): ATT = {fe.params['post_late']:.4f}")
print()
print("Interpretation: if early adopter states had larger effects, it could indicate")
print("novelty effects (initial behavior change that fades) or lagged market maturation.")

## Alcohol-impaired vs drug-impaired fatalities

In [ ]:
for outcome in ['alcohol_fatalities_per_100k','drug_fatalities_per_100k']:
    if outcome not in sub.columns:
        print(f"  {outcome} not available — add to FARS build step")
        continue
    fe_o = PanelOLS(
        idx[outcome],
        idx[['post']],
        entity_effects=True, time_effects=True,
    ).fit(cov_type='clustered', cluster_entity=True)
    b  = fe_o.params['post']
    ci = fe_o.conf_int().loc['post']
    print(f"{outcome:<40}  ATT = {b:+.4f}  [{ci['lower']:+.4f}, {ci['upper']:+.4f}]")